In [194]:
import pandas as pd
import numpy as np

from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import train_test_split, GridSearchCV, ShuffleSplit
from sklearn.preprocessing import OrdinalEncoder, OneHotEncoder
from sklearn.pipeline import Pipeline


### Loading Data

In [195]:
data=pd.read_csv('nursery.data', sep=',', header=None)
data.columns=['parents', 'has_nurs', 'form', 'children', 'housing',
              'finance', 'social', 'health', 'class']
data.head()

,parents,has_nurs,form,children,housing,finance,social,health,class
0,usual,proper,complete,1,convenient,convenient,nonprob,recommended,recommend
1,usual,proper,complete,1,convenient,convenient,nonprob,priority,priority
2,usual,proper,complete,1,convenient,convenient,nonprob,not_recom,not_recom
3,usual,proper,complete,1,convenient,convenient,slightly_prob,recommended,recommend
4,usual,proper,complete,1,convenient,convenient,slightly_prob,priority,priority


In [196]:
data['class'].value_counts()

class
not_recom     4320
priority      4266
spec_prior    4044
very_recom     328
recommend        2
Name: count, dtype: int64

We can see that the class `spec_prior` and `very_recom` are not present in the documentation and `recommend` only has two instances. Therefore we merge these classes into one and call it `recommend` as it allows us to stay consistent with the documentation.

In [197]:
data.loc[data['class'].isin(['spec_prior', 'very_recom']) , 'class']='recommend'
data['class'].value_counts()

class
recommend    4374
not_recom    4320
priority     4266
Name: count, dtype: int64

# Task 1

The data is split into `temp` and `test`. And `temp` is further split into `train` and `validation` subsets. The latter is done through `ShuffleSplit` instance that only makes one split. This way of splitting allows for use of GridSearchCV that helps in tuning the hyperparameters of the models.

In [198]:
def split_test_data(X, y, random_state=None):
    X_temp, X_test, y_temp, y_test=train_test_split(X, y, test_size=0.1, random_state=random_state)
    return X_temp, X_test, y_temp, y_test

In [199]:
X, y=data.drop('class', axis=1), data['class']
X_temp, X_test, y_temp, y_test=split_test_data(X, y, random_state=1)

### Decision Tree (categorical features encoded using label encoder)

Function to tune the Decision Tree pipeline. The function return the grid search instance that contains the best hyperparameters and tuned classifier.

In [200]:
def tune_dt(X, y, model, random_state=None):
    valid_splitter=ShuffleSplit(n_splits=1, test_size=0.2, random_state=random_state)
    gscv=GridSearchCV(model, param_grid={'clsf__criterion': ['gini', 'entropy', 'log_loss'],
                                                'clsf__max_features': [None, 'sqrt', 'log2'],
                                                'clsf__max_depth': [None] + list(range(1, 20, 2))
                                        },
                        n_jobs=-1, cv=valid_splitter)
    gscv.fit(X, y)
    return gscv


Pipeline that encodes the data and fits the Decision Tree

In [201]:
pipeline=Pipeline(steps=[('encoder', OrdinalEncoder()),
                         ('clsf', DecisionTreeClassifier(random_state=1))])

In [202]:
gscv=tune_dt(X_temp, y_temp, model=pipeline, random_state=1)

In [203]:
gscv.best_params_, gscv.best_score_

({'clsf__criterion': 'gini',
  'clsf__max_depth': None,
  'clsf__max_features': None},
 np.float64(0.9888555507929704))

### Decision Tree (categorical features encoded using OneHotEncoder)

Pipeline that encodes the data and fits the Decision Tree

In [204]:
pipeline=Pipeline(steps=[('encoder', OneHotEncoder()),
                         ('clsf', DecisionTreeClassifier(random_state=1))])

In [205]:
gscv=tune_dt(X_temp, y_temp, model=pipeline, random_state=1)

In [206]:
gscv.best_params_, gscv.best_score_

({'clsf__criterion': 'entropy',
  'clsf__max_depth': None,
  'clsf__max_features': None},
 np.float64(0.994856408058294))

### Logistic Regression (with L1 regularization)

Function to tune the Logistic Regression pipeline. The function return the grid search instance that contains the best hyperparameters and tuned classifier.

In [208]:
def tune_lr(X, y, model, random_state=None):
    valid_splitter=ShuffleSplit(n_splits=1, test_size=0.2, random_state=random_state)
    gscv=GridSearchCV(model, param_grid={'clsf__C': np.linspace(0.1, 1, 10),
                                                'clsf__fit_intercept': [True, False],
                                                'clsf__solver': ['saga'],
                                                'clsf__intercept_scaling': np.linspace(1, 5, 5),
                                                'clsf__max_iter': [100, 200, 300]
                                        },
                        n_jobs=-1, cv=valid_splitter)
    gscv.fit(X, y)
    return gscv


Pipeline that encodes the data and fits the Logistic Regression

In [209]:
pipeline=Pipeline(steps=[('encoder', OneHotEncoder()),
                         ('clsf', LogisticRegression(penalty='l1', random_state=1))])

In [210]:
gscv=tune_lr(X_temp, y_temp, model=pipeline, random_state=1)

d:\msc\workspace\dal\.venv\lib\site-packages\sklearn\linear_model\_sag.py:349: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(


In [193]:
gscv.best_params_, gscv.best_score_

({'clsf__C': np.float64(0.4),
  'clsf__fit_intercept': True,
  'clsf__intercept_scaling': np.float64(1.0),
  'clsf__max_iter': 100,
  'clsf__solver': 'saga'},
 np.float64(0.9082726103729104))

### KNN

Function to tune the KNN pipeline. The function returns the grid search instance that contains the best hyperparameters and tuned classifier.

In [237]:
def tune_knn(X, y, model, random_state=None):
    valid_splitter=ShuffleSplit(n_splits=1, test_size=0.2, random_state=random_state)
    gscv=GridSearchCV(model, param_grid={'clsf__n_neighbors': [5, 7, 11, 13, 15],
                                                'clsf__weights': ['uniform', 'distance'],
                                        },
                        n_jobs=-1, cv=valid_splitter)
    gscv.fit(X, y)
    return gscv


Pipeline that encodes the data and fits the KNN Classifier

In [238]:
pipeline=Pipeline(steps=[('encoder', OneHotEncoder(sparse_output=False)),
                         ('clsf', KNeighborsClassifier())])

In [239]:
gscv=tune_knn(X_temp, y_temp, model=pipeline, random_state=1)

In [240]:
gscv.best_params_, gscv.best_score_

({'clsf__n_neighbors': 11, 'clsf__weights': 'uniform'},
 np.float64(0.9682811830261466))